# PaCMAP visualization of ModernMolBERT embeddings

In [1]:
import json
import re
from pathlib import Path

from tqdm.auto import tqdm

import colorcet as cc
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import numpy as np
import pacmap
import pandas as pd
import pyarrow.parquet as pq

import datashader as ds
import datashader.transfer_functions as tf
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
import matplotlib.colors as mcolors

/Users/skn506/Documents/ModernMolBERT/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config

In [4]:
EMBED_DIR = Path("../../outputs/visualize/best_span_100k")

FIGURE_DIR = Path("../../figures/pacmap")

# Source parquet used when embedding - needed to join extra property columns.
SOURCE_PARQUET = Path("../../data/pretrain/chembl36_selfies/train.parquet")

# Column from metadata.parquet to color by.
COLOR_COLUMN = "alogp"

# Additional property columns to join from SOURCE_PARQUET.
# Any subset of the list below - they are merged by chembl_id at load time.
EXTRA_COLUMNS = [
    "psa",
    "hba",
    "hbd",
    "qed_weighted",
    "mw_freebase",
    "rtb",
    "aromatic_rings",
    "heavy_atoms",
    "num_ro5_violations",
]

# Display labels for the color legend.
COLUMN_LABELS = {
    "alogp": "ALogP (lipophilicity)",
    "psa": "Polar surface area (Å²)",
    "hba": "H-bond acceptors",
    "hbd": "H-bond donors",
    "qed_weighted": "QED (drug-likeness)",
    "mw_freebase": "Molecular weight (Da)",
    "rtb": "Rotatable bonds",
    "aromatic_rings": "Aromatic rings",
    "heavy_atoms": "Heavy atom count",
    "num_ro5_violations": "Ro5 violations",
    "is_valid": "Valid SELFIES",
}

# Sweep all labeled measures that are available after loading and joining metadata.
MEASURE_COLUMNS = list(COLUMN_LABELS)

# Expanded grid around best setting (n_neighbors=30, MN_ratio=0.0, FP_ratio=12.0)
# Try more neighbors and some finer MN/FP ratios for more separation
N_NEIGHBORS_GRID = [20, 30, 40, 50]
MN_RATIO_GRID = [0.0, 0.25, 0.5]
FP_RATIO_GRID = [8.0, 10.0, 12.0, 15.0]
PACMAP_PARAMETER_GRID = [
    {"n_neighbors": n_neighbors, "MN_ratio": mn_ratio, "FP_ratio": fp_ratio}
    for n_neighbors in N_NEIGHBORS_GRID
    for mn_ratio in MN_RATIO_GRID
    for fp_ratio in FP_RATIO_GRID
]

# Fixed setting for sweeping all measures without rerunning the full parameter grid.
SET_PACMAP_PARAMS = {"n_neighbors": 40, "MN_ratio": 0.25, "FP_ratio": 12.0}
SET_PACMAP_PLOT_SIZES = ["small"]

## Load

In [5]:
embeddings = np.load(EMBED_DIR / "embeddings.npy")
meta = pd.read_parquet(EMBED_DIR / "metadata.parquet")

print(f"Embeddings: {embeddings.shape}")
print(f"Metadata columns: {meta.columns.tolist()}")

Embeddings: (100000, 512)
Metadata columns: ['chembl_id', 'smiles_canonical_clean', 'selfies', 'alogp', 'canonical_smiles', 'standard_inchi_key', 'is_valid', 'embedding_row']


In [6]:
if EXTRA_COLUMNS and SOURCE_PARQUET.exists():
    source_columns = set(pq.ParquetFile(SOURCE_PARQUET).schema.names)

    cols_to_load = []
    for c in EXTRA_COLUMNS:
        if c not in source_columns:
            continue

        if c not in meta.columns:
            cols_to_load.append(c)
            continue

        # Reload column if present but unusable
        numeric = pd.to_numeric(meta[c], errors="coerce")
        if numeric.notna().sum() == 0:
            cols_to_load.append(c)

    missing_source_cols = [
        c for c in EXTRA_COLUMNS if c not in meta.columns and c not in source_columns
    ]

    if cols_to_load:
        # Drop stale columns before merging fresh values
        meta = meta.drop(columns=[c for c in cols_to_load if c in meta.columns])

        props = pd.read_parquet(
            SOURCE_PARQUET,
            columns=["chembl_id"] + cols_to_load,
        )

        meta["chembl_id"] = meta["chembl_id"].astype(str).str.strip()
        props["chembl_id"] = props["chembl_id"].astype(str).str.strip()

        meta = meta.merge(props, on="chembl_id", how="left", validate="many_to_one")

        print(f"Joined/reloaded {cols_to_load} from source parquet.")

    if missing_source_cols:
        print(f"Warning: missing source columns: {missing_source_cols}")

elif EXTRA_COLUMNS:
    print(f"Warning: SOURCE_PARQUET not found at {SOURCE_PARQUET}")

Joined/reloaded ['psa', 'hba', 'hbd', 'qed_weighted', 'mw_freebase', 'rtb', 'aromatic_rings', 'heavy_atoms', 'num_ro5_violations'] from source parquet.


## Run PaCMAP

In [7]:
reducer = pacmap.PaCMAP(
    n_components=2,
    n_neighbors=5,
    MN_ratio=0.0,
    FP_ratio=12.0,
    verbose=True,
)

coords = reducer.fit_transform(embeddings, init="pca")
print(f"PaCMAP output: {coords.shape}")

Applied PCA, the dimensionality becomes 100
PaCMAP(n_neighbors=5, n_MN=0, n_FP=60, distance=euclidean, lr=1.0, n_iters=(100, 100, 250), apply_pca=True, opt_method='adam', verbose=True, intermediate=False, seed=None, knn_backend=faiss)
Finding pairs
Found nearest neighbor
Calculated sigma
Found scaled dist
Pairs sampled successfully.
((500000, 2), (0, 2), (6000000, 2))
Initial Loss: 3090885.75
Iteration:   10, Loss: 537784.625000
Iteration:   20, Loss: 476447.156250
Iteration:   30, Loss: 446217.000000
Iteration:   40, Loss: 427591.500000
Iteration:   50, Loss: 415485.562500
Iteration:   60, Loss: 407266.656250
Iteration:   70, Loss: 401199.187500
Iteration:   80, Loss: 396500.187500
Iteration:   90, Loss: 392663.656250
Iteration:  100, Loss: 389632.312500
Iteration:  110, Loss: 542485.875000
Iteration:  120, Loss: 536391.937500
Iteration:  130, Loss: 531851.625000
Iteration:  140, Loss: 528533.312500
Iteration:  150, Loss: 526082.187500
Iteration:  160, Loss: 524152.687500
Iteration:  

## Visualize

In [8]:
available_measures = [c for c in MEASURE_COLUMNS if c in meta.columns and meta[c].notna().any()]
missing_measures = [c for c in MEASURE_COLUMNS if c not in meta.columns]
empty_measures = [c for c in MEASURE_COLUMNS if c in meta.columns and not meta[c].notna().any()]
print(f"Available measures: {available_measures}")
if missing_measures:
    print(f"Missing measures skipped: {missing_measures}")
if empty_measures:
    print(f"Empty measures skipped: {empty_measures}")


def _slug(value):
    text = str(value).replace(".", "p")
    return re.sub(r"[^A-Za-z0-9]+", "_", text).strip("_").lower()


# Measures that should use a sequential Colorcet colormap.
SEQUENTIAL_MEASURES = [
    "mw_freebase",
    "heavy_atoms",
    "aromatic_rings",
    "rtb",
    "num_ro5_violations",
    "psa",
    "hba",
    "hbd",
    "qed_weighted",
    "alogp",
]
COLORCET_SEQUENTIAL_CMAP = "CET_L17"
COLORCET_DIVERGENT_CMAP = "CET_D1A"
COLORCET_CATEGORICAL_PALETTE = "glasbey"
FIGURE_SPECS = {
    "small": {
        "figsize": (3, 3),
        "dpi": 300,
        "point_size": 2,
        "point_alpha": 0.86,
        "datashader_spread_px": 3,
        "datashader_spread_shape": "circle",
        "title_size": 8.5,
        "label_size": 8,
        "colorbar_label_size": 5.5,
        "tick_label_size": 5,
        "legend_fontsize": 5.5,
        "legend_title_fontsize": 6,
        "legend_marker_size": 3.8,
        "legend_max": 8,
        "colorbar_width": "3%",
        "colorbar_height": "24%",
        "colorbar_borderpad": 0.7,
    },
    "large": {
        "figsize": (9.0, 7.5),
        "dpi": 300,
        "point_size": 3.2,
        "point_alpha": 0.84,
        "datashader_spread_px": 2,
        "datashader_spread_shape": "circle",
        "title_size": 14,
        "label_size": 11,
        "colorbar_label_size": 8,
        "tick_label_size": 7,
        "legend_fontsize": 7,
        "legend_title_fontsize": 8,
        "legend_marker_size": 5.5,
        "legend_max": 14,
        "colorbar_width": "2.2%",
        "colorbar_height": "28%",
        "colorbar_borderpad": 0.9,
    },
}
DEFAULT_FIGURE_SIZE = "small"


def _colorcet_cmap(name, fallback):
    if hasattr(cc.cm, name):
        return getattr(cc.cm, name)
    try:
        return cc.cm[name]
    except (KeyError, TypeError):
        return plt.get_cmap(fallback)


def _colorcet_palette(name, n_colors):
    palette = getattr(cc, name, None)
    if palette is None:
        palette = getattr(cc, "palette", {}).get(name)
    if palette is None:
        cmap = plt.get_cmap("tab20", max(n_colors, 1))
        return [cmap(i / max(n_colors - 1, 1)) for i in range(n_colors)]
    palette = list(palette)
    return [palette[i % len(palette)] for i in range(n_colors)]


def pacmap_run_id(params):
    return (
        f"nn{params['n_neighbors']:03d}_mn{_slug(params['MN_ratio'])}_fp{_slug(params['FP_ratio'])}"
    )


def _spread_datashader_image(img, figure_spec):
    spread_px = figure_spec.get("datashader_spread_px", 0)
    if not spread_px:
        return img
    return tf.spread(
        img,
        px=spread_px,
        shape=figure_spec.get("datashader_spread_shape", "circle"),
    )


def plot_pacmap_measure(
    coords,
    meta,
    color_column,
    *,
    params=None,
    output_path=None,
    show=True,
    plot_size=DEFAULT_FIGURE_SIZE,
    legend_max=None,
    legend_loc="lower right",
    legend_bbox=(0.98, 0.02),
    sequential_measures=SEQUENTIAL_MEASURES,
    sequential_cmap=COLORCET_SEQUENTIAL_CMAP,
    divergent_cmap=COLORCET_DIVERGENT_CMAP,
    categorical_palette=COLORCET_CATEGORICAL_PALETTE,
):
    if color_column not in meta.columns:
        raise ValueError(f"{color_column!r} not in metadata. Available: {meta.columns.tolist()}")

    color_series = meta[color_column]
    is_numeric = pd.api.types.is_numeric_dtype(color_series) and not pd.api.types.is_bool_dtype(
        color_series
    )
    col_label = COLUMN_LABELS.get(color_column, color_column) or str(color_column)
    valid_coords = np.isfinite(coords).all(axis=1)

    if plot_size not in FIGURE_SPECS:
        raise ValueError(f"Unknown plot_size {plot_size!r}. Use one of {list(FIGURE_SPECS)}")
    figure_spec = FIGURE_SPECS[plot_size]
    legend_max = figure_spec["legend_max"] if legend_max is None else legend_max

    fig, ax = plt.subplots(figsize=figure_spec["figsize"], dpi=figure_spec["dpi"])

    plot_width = int(figure_spec["figsize"][0] * figure_spec["dpi"])
    plot_height = int(figure_spec["figsize"][1] * figure_spec["dpi"])

    if is_numeric:
        numeric_values = pd.to_numeric(color_series, errors="coerce")
        mask = valid_coords & numeric_values.notna().to_numpy()
        if not mask.any():
            raise ValueError(f"No finite values available for {color_column!r}")

        df = pd.DataFrame(coords[mask], columns=["x", "y"])
        df["val"] = numeric_values.to_numpy()[mask]

        # Winsorize to [1%, 99%] to reduce outlier influence on color scale.
        q_low = df["val"].quantile(0.01)
        q_high = df["val"].quantile(0.99)
        df["val"] = df["val"].clip(lower=q_low, upper=q_high)

        xmin, xmax = df["x"].min(), df["x"].max()
        ymin, ymax = df["y"].min(), df["y"].max()
        x_pad, y_pad = (xmax - xmin) * 0.05, (ymax - ymin) * 0.05
        x_range, y_range = (xmin - x_pad, xmax + x_pad), (ymin - y_pad, ymax + y_pad)

        cvs = ds.Canvas(
            plot_width=plot_width,
            plot_height=plot_height,
            x_range=x_range,
            y_range=y_range,
        )
        agg = cvs.points(df, "x", "y", ds.mean("val"))

        cmap_name = sequential_cmap if color_column in sequential_measures else divergent_cmap
        cmap = _colorcet_cmap(cmap_name, "viridis").reversed()

        img = _spread_datashader_image(tf.shade(agg, cmap=cmap, how="linear"), figure_spec)
        ax.imshow(
            img.to_pil(),
            extent=[x_range[0], x_range[1], y_range[0], y_range[1]],
            origin="lower",
        )

        cax = inset_axes(
            ax,
            width=figure_spec["colorbar_width"],
            height=figure_spec["colorbar_height"],
            loc="lower right",
            borderpad=figure_spec["colorbar_borderpad"],
        )
        norm = Normalize(vmin=q_low, vmax=q_high)
        sm = ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        cb = fig.colorbar(sm, cax=cax)
        cb.ax.set_title(str(col_label), fontsize=figure_spec["colorbar_label_size"], pad=3)
        cb.ax.tick_params(labelsize=figure_spec["tick_label_size"])
    else:
        mask = valid_coords & color_series.notna().to_numpy()
        if not mask.any():
            raise ValueError(f"No finite values available for {color_column!r}")

        categories = color_series[mask].astype(str)
        unique_cats = sorted(pd.unique(categories))
        palette = _colorcet_palette(categorical_palette, len(unique_cats))
        cat_to_int = {c: i for i, c in enumerate(unique_cats)}

        df = pd.DataFrame(coords[mask], columns=["x", "y"])
        df["cat"] = pd.Categorical(categories)

        xmin, xmax = df["x"].min(), df["x"].max()
        ymin, ymax = df["y"].min(), df["y"].max()
        x_pad, y_pad = (xmax - xmin) * 0.05, (ymax - ymin) * 0.05
        x_range, y_range = (xmin - x_pad, xmax + x_pad), (ymin - y_pad, ymax + y_pad)

        cvs = ds.Canvas(
            plot_width=plot_width,
            plot_height=plot_height,
            x_range=x_range,
            y_range=y_range,
        )
        agg = cvs.points(df, "x", "y", ds.count_cat("cat"))

        color_key = {c: mcolors.to_hex(palette[cat_to_int[c]]) for c in unique_cats}
        img = _spread_datashader_image(
            tf.shade(agg, color_key=color_key, how="eq_hist"), figure_spec
        )
        ax.imshow(
            img.to_pil(),
            extent=[x_range[0], x_range[1], y_range[0], y_range[1]],
            origin="lower",
        )

        handles = [
            Line2D(
                [0],
                [0],
                marker="o",
                color="w",
                markerfacecolor=palette[i],
                markersize=figure_spec["legend_marker_size"],
                label=str(c),
            )
            for i, c in enumerate(unique_cats[:legend_max])
        ]
        ax.legend(
            handles=handles,
            title=str(col_label),
            bbox_to_anchor=legend_bbox,
            loc=legend_loc,
            fontsize=figure_spec["legend_fontsize"],
            title_fontsize=figure_spec["legend_title_fontsize"],
            frameon=True,
            framealpha=0.72,
            borderpad=0.25,
            labelspacing=0.25,
            handlelength=0.8,
        )

    title = f"PaCMAP - {col_label}"
    ax.set_title(title, fontsize=figure_spec["title_size"], pad=8)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_visible(False)

    fig.subplots_adjust(left=0.02, right=0.98, bottom=0.02, top=0.92)
    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, dpi=figure_spec["dpi"], facecolor="white")
    if show:
        plt.show()
    else:
        plt.close(fig)
    return fig


def sweep_pacmap_measures_for_params(
    params,
    *,
    measures=None,
    plot_sizes=None,
    output_dir=None,
    run_coords=None,
    verbose=True,
):
    measures = available_measures if measures is None else list(measures)
    plot_sizes = SET_PACMAP_PLOT_SIZES if plot_sizes is None else list(plot_sizes)
    output_dir = FIGURE_DIR / "set_parameters" if output_dir is None else Path(output_dir)

    unknown_sizes = [plot_size for plot_size in plot_sizes if plot_size not in FIGURE_SPECS]
    if unknown_sizes:
        raise ValueError(f"Unknown plot sizes: {unknown_sizes}. Use one of {list(FIGURE_SPECS)}")

    run_id = pacmap_run_id(params)
    if run_coords is None:
        if verbose:
            print(f"Running {run_id}: {params}")
        reducer = pacmap.PaCMAP(n_components=2, verbose=verbose, **params)
        run_coords = reducer.fit_transform(embeddings, init="pca")

    records = []
    plot_jobs = [(measure, plot_size) for measure in measures for plot_size in plot_sizes]
    for measure, plot_size in tqdm(
        plot_jobs,
        desc=f"Plotting {run_id}",
        disable=not verbose,
    ):
        output_path = output_dir / plot_size / f"{run_id}_{_slug(measure)}_{plot_size}.png"
        plot_pacmap_measure(
            run_coords,
            meta,
            measure,
            params=params,
            output_path=output_path,
            show=False,
            plot_size=plot_size,
        )
        records.append(
            {
                "run_id": run_id,
                "measure": measure,
                "plot_size": plot_size,
                "figure_path": str(output_path),
                **params,
            }
        )

    manifest = pd.DataFrame(records)
    manifest_path = output_dir / f"{run_id}_measure_sweep_manifest.csv"
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest.to_csv(manifest_path, index=False)

    config_path = output_dir / f"{run_id}_measure_sweep_config.json"
    config_path.write_text(
        json.dumps(
            {
                "embedding_dir": str(EMBED_DIR),
                "source_parquet": str(SOURCE_PARQUET),
                "figure_dir": str(output_dir),
                "params": params,
                "measures": measures,
                "plot_sizes": plot_sizes,
                "missing_measures": missing_measures,
                "empty_measures": empty_measures,
                "figure_specs": FIGURE_SPECS,
            },
            indent=2,
        ),
        encoding="utf-8",
    )

    if verbose:
        print(f"Wrote {len(manifest)} figures to {output_dir}")
        print(f"Wrote manifest: {manifest_path}")
        print(f"Wrote config: {config_path}")
    return manifest

Available measures: ['alogp', 'psa', 'hba', 'hbd', 'qed_weighted', 'mw_freebase', 'rtb', 'aromatic_rings', 'heavy_atoms', 'num_ro5_violations', 'is_valid']


## Sweep measures for fixed PaCMAP parameters

In [ ]:
fixed_measure_manifest = sweep_pacmap_measures_for_params(
    SET_PACMAP_PARAMS,
    plot_sizes=SET_PACMAP_PLOT_SIZES,
)
fixed_measure_manifest

Running nn040_mn0p25_fp12p0: {'n_neighbors': 40, 'MN_ratio': 0.25, 'FP_ratio': 12.0}
Applied PCA, the dimensionality becomes 100
PaCMAP(n_neighbors=40, n_MN=10, n_FP=480, distance=euclidean, lr=1.0, n_iters=(100, 100, 250), apply_pca=True, opt_method='adam', verbose=True, intermediate=False, seed=None, knn_backend=faiss)
Finding pairs
Found nearest neighbor
Calculated sigma
Found scaled dist
Pairs sampled successfully.
((4000000, 2), (1000000, 2), (48000000, 2))
Initial Loss: 9234991.0
Iteration:   10, Loss: 8393638.000000
Iteration:   20, Loss: 6623204.500000
Iteration:   30, Loss: 6289727.000000
Iteration:   40, Loss: 6017600.000000
Iteration:   50, Loss: 5734777.000000
Iteration:   60, Loss: 5507214.000000


## Sweep PaCMAP parameters

In [ ]:
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

sweep_records = []
parameter_iter = tqdm(PACMAP_PARAMETER_GRID, desc="PaCMAP parameter grid")
for run_index, params in enumerate(parameter_iter, start=1):
    run_id = pacmap_run_id(params)
    parameter_iter.set_postfix_str(run_id)
    tqdm.write(f"[{run_index:02d}/{len(PACMAP_PARAMETER_GRID)}] Running {run_id}: {params}")
    reducer = pacmap.PaCMAP(n_components=2, verbose=False, **params)
    sweep_coords = reducer.fit_transform(embeddings, init="pca")

    plot_jobs = [
        (measure, plot_size) for measure in available_measures for plot_size in FIGURE_SPECS
    ]
    for measure, plot_size in tqdm(plot_jobs, desc=f"Plotting {run_id}", leave=False):
        output_path = FIGURE_DIR / plot_size / f"{run_id}_{_slug(measure)}_{plot_size}.png"
        plot_pacmap_measure(
            sweep_coords,
            meta,
            measure,
            params=params,
            output_path=output_path,
            show=False,
            plot_size=plot_size,
        )
        sweep_records.append(
            {
                "run_id": run_id,
                "measure": measure,
                "plot_size": plot_size,
                "figure_path": str(output_path),
                **params,
            }
        )

manifest = pd.DataFrame(sweep_records)
manifest_path = FIGURE_DIR / "pacmap_sweep_manifest.csv"
manifest.to_csv(manifest_path, index=False)

config_path = FIGURE_DIR / "pacmap_sweep_config.json"
config_path.write_text(
    json.dumps(
        {
            "embedding_dir": str(EMBED_DIR),
            "source_parquet": str(SOURCE_PARQUET),
            "figure_dir": str(FIGURE_DIR),
            "n_parameter_combinations": len(PACMAP_PARAMETER_GRID),
            "parameter_grid": PACMAP_PARAMETER_GRID,
            "measures": available_measures,
            "missing_measures": missing_measures,
            "empty_measures": empty_measures,
            "figure_specs": FIGURE_SPECS,
        },
        indent=2,
    ),
    encoding="utf-8",
)

print(f"Wrote {len(manifest)} figures to {FIGURE_DIR}")
print(f"Wrote manifest: {manifest_path}")
print(f"Wrote config: {config_path}")

PaCMAP parameter grid:   0%|          | 0/48 [00:00<?, ?it/s, nn020_mn0p0_fp8p0]

[01/48] Running nn020_mn0p0_fp8p0: {'n_neighbors': 20, 'MN_ratio': 0.0, 'FP_ratio': 8.0}
